# Spreadsheet

    Make a spreadsheet using pinkfish.  This is useful for developing trading strategies.
    It can also be used as a tool for buy and sell signals that you then manually execute.

In [1]:
import datetime

import matplotlib.pyplot as plt
import pandas as pd

from talib.abstract import *

import pinkfish as pf
import pinkfish.itable as itable

# Format price data
pd.options.display.float_format = '{:0.2f}'.format

%matplotlib inline

In [2]:
# Set size of inline plots
'''note: rcParams can't be in same cell as import matplotlib
   or %matplotlib inline
   
   %matplotlib notebook: will lead to interactive plots embedded within
   the notebook, you can zoom and resize the figure
   
   %matplotlib inline: only draw static images in the notebook
'''
plt.rcParams["figure.figsize"] = (10, 7)

Some global data

In [3]:
symbol = 'SPY'
start = datetime.datetime(1900, 1, 1)
end = datetime.datetime.now()

Fetch symbol data from internet; do not use local cache. 

In [4]:
ts = pf.fetch_timeseries(symbol, use_cache=False)
ts.tail()

,open,high,low,close,adj_close,volume
date,,,,,,
2026-06-08,743.36,745.34,738.19,739.22,739.22,49319100
2026-06-09,743.63,746.90,722.59,737.05,737.05,87683500
2026-06-10,733.39,738.38,725.33,725.43,725.43,60341300
2026-06-11,728.76,740.00,724.41,737.76,737.76,86330500
2026-06-12,740.71,744.44,735.03,741.75,741.75,56939800


Select timeseries between start and end.  Back adjust prices relative to adj_close for dividends and splits.

In [5]:
ts = pf.select_tradeperiod(ts, start, end, use_adj=False)
ts.head()

,open,high,low,close,adj_close,volume
date,,,,,,
1993-01-29,43.97,43.97,43.75,43.94,24.18,1003200
1993-02-01,43.97,44.25,43.97,44.25,24.35,480500
1993-02-02,44.22,44.38,44.12,44.34,24.40,201300
1993-02-03,44.41,44.84,44.38,44.81,24.66,529400
1993-02-04,44.97,45.09,44.47,45.00,24.76,531500


Add technical indicators

In [6]:
# Add 200 day MA.
ts['sma200'] = pf.SMA(ts, timeperiod=200)

# Add ATR.
ts['atr'] = ATR(ts, timeperiod=14)

# Add 5 day high, and 5 day low
ts['high5'] = pd.Series(ts.high).rolling(window=5).max()
ts['low5'] = pd.Series(ts.low).rolling(window=5).min()

# Add RSI, and 2-period cumulative RSI
ts['rsi2'] = RSI(ts, timeperiod=2)
ts['c2rsi2'] = pd.Series(ts.rsi2).rolling(window=2).sum()

# Add midpoint
ts['mp'] = (ts.high + ts.low) / 2

# Add 10 day SMA of midpoint
ts['sma10'] = pd.Series(ts.mp).rolling(window=10).mean()

# Add temporary rolling 10 day Standard Deviation of midpoint
ts['__sd__'] = pd.Series(ts.mp).rolling(window=10).std()

# Add standard deviation envelope or channel around midpoint
ts['upper'] = ts.sma10 + ts['__sd__']*2
ts['lower'] = ts.sma10 - ts['__sd__']*2

# Drop temporary columns.
ts.drop(columns=['__sd__'], inplace=True)

Finalize timeseries

In [7]:
ts, start = pf.finalize_timeseries(ts, start, dropna=True)
ts.tail()

,open,high,low,close,adj_close,volume,sma200,atr,high5,low5,rsi2,c2rsi2,mp,sma10,upper,lower
date,,,,,,,,,,,,,,,,
2026-06-08,743.36,745.34,738.19,739.22,739.22,49319100,684.43,7.63,760.40,735.53,19.87,27.96,741.77,752.17,763.56,740.78
2026-06-09,743.63,746.90,722.59,737.05,737.05,87683500,684.94,8.82,758.80,722.59,14.90,34.77,734.75,750.62,766.51,734.73
2026-06-10,733.39,738.38,725.33,725.43,725.43,60341300,685.34,9.12,758.31,722.59,4.05,18.96,731.86,748.83,768.68,728.97
2026-06-11,728.76,740.00,724.41,737.76,737.76,86330500,685.82,9.58,752.82,722.59,62.30,66.35,732.20,746.83,769.06,724.59
2026-06-12,740.71,744.44,735.03,741.75,741.75,56939800,686.30,9.57,746.90,722.59,72.93,135.23,739.74,745.16,766.70,723.63


Select a smaller time from for use with itable

In [8]:
df = ts['2025-06-01':]
df.head()

,open,high,low,close,adj_close,volume,sma200,atr,high5,low5,rsi2,c2rsi2,mp,sma10,upper,lower
date,,,,,,,,,,,,,,,,
2025-06-02,587.76,592.79,585.06,592.71,585.99,61630500,577.22,9.15,593.20,578.43,82.82,140.96,588.92,587.41,595.47,579.35
2025-06-03,592.34,597.08,591.85,596.09,589.33,63606200,577.49,8.87,597.08,583.24,92.19,175.01,594.46,587.67,596.51,578.83
2025-06-04,596.96,597.95,595.49,595.93,589.18,57314200,577.70,8.42,597.95,583.24,87.66,179.85,596.72,588.16,598.45,577.87
2025-06-05,597.63,599.00,591.05,593.05,586.33,92278700,577.89,8.38,599.00,583.24,31.67,119.33,595.02,588.94,600.06,577.82
2025-06-06,598.66,600.83,596.86,599.14,592.35,66588700,578.09,8.34,600.83,585.06,81.54,113.21,598.85,590.43,602.53,578.32


Use itable to format the spreadsheet.  New 5 day high has blue highlight; new 5 day low has red highlight.

In [9]:
pt = itable.PrettyTable(
    df, tstyle=itable.TableStyle(theme='theme1'), header_row=True, rpt_header=20)

# pt = itable.PrettyTable(
#      df, tstyle=itable.TableStyle(theme='theme1'), header_row=True, rpt_header=20)

pt.update_col_header_style(
    format_function=lambda x: x.upper(), text_align='right')
pt.update_row_header_style(
    format_function=lambda x: pd.to_datetime(str(x)).strftime('%Y/%m/%d'), text_align='right')

for col in range(pt.num_cols):
    if pt.df.columns[col] == 'volume':
        pt.update_cell_style(cols=[col], format_function=lambda x: format(x, '.0f'), text_align='right')
    else:
        pt.update_cell_style(cols=[col], format_function=lambda x: format(x, '.2f'), text_align='right')

for row in range(pt.num_rows):
    if row == 0:
        continue
    if (pt.df['high5'].iloc[row] == pt.df['high'].iloc[row]) and \
       (pt.df['high5'].iloc[row] > pt.df['high'].iloc[row-1]):
        col = df.columns.get_loc('high5')    
        pt.update_cell_style(rows=[row], cols=[col], color='blue')
    if (pt.df['low5'].iloc[row] == pt.df['low'].iloc[row]) and \
       (pt.df['low5'].iloc[row] < pt.df['low'].iloc[row-1]):
        col = df.columns.get_loc('low5')
        pt.update_cell_style(rows=[row], cols=[col], color='maroon')          

In [10]:
pt

,OPEN,HIGH,LOW,CLOSE,ADJ_CLOSE,VOLUME,SMA200,ATR,HIGH5,LOW5,RSI2,C2RSI2,MP,SMA10,UPPER,LOWER
2025/06/02,587.76,592.79,585.06,592.71,585.99,61630500,577.22,9.15,593.20,578.43,82.82,140.96,588.92,587.41,595.47,579.35
2025/06/03,592.34,597.08,591.85,596.09,589.33,63606200,577.49,8.87,597.08,583.24,92.19,175.01,594.46,587.67,596.51,578.83
2025/06/04,596.96,597.95,595.49,595.93,589.18,57314200,577.70,8.42,597.95,583.24,87.66,179.85,596.72,588.16,598.45,577.87
2025/06/05,597.63,599.00,591.05,593.05,586.33,92278700,577.89,8.38,599.00,583.24,31.67,119.33,595.02,588.94,600.06,577.82
2025/06/06,598.66,600.83,596.86,599.14,592.35,66588700,578.09,8.34,600.83,585.06,81.54,113.21,598.85,590.43,602.53,578.32
2025/06/09,599.72,601.25,598.49,599.68,592.88,53016400,578.30,7.94,601.25,591.05,83.65,165.19,599.87,592.54,602.80,582.28
2025/06/10,600.22,603.47,599.09,603.08,596.24,66247000,578.51,7.69,603.47,591.05,93.31,176.96,601.28,594.18,604.24,584.13
2025/06/11,604.19,605.06,599.27,601.36,594.54,73658200,578.73,7.55,605.06,591.05,58.40,151.71,602.17,595.41,606.11,584.71
2025/06/12,600.01,603.75,599.52,603.75,596.91,64129000,578.94,7.31,605.06,596.86,79.61,138.01,601.64,596.61,607.12,586.10
2025/06/13,598.50,601.85,595.48,597.00,590.23,89506000,579.12,7.38,605.06,595.48,20.52,100.13,598.66,597.76,605.94,589.57


In [11]:
pf.get_quote([symbol])

{'SPY': 741.75}